# Relational E-Commerce Analysis

## Goal

Understand how aggregation and joins change table grain, combine order-level
and item-level data safely, validate key relationships, and produce reliable
order-level metrics.

## Table grain

- `orders`: one row represents one order.
- `order_items`: one row represents one item line within an order.
- `order_value`: one row represents one order after aggregating its item rows.
- `orders_enriched`: one row represents one order with calculated item metrics.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    """
    Walk upward from the current working directory until we find
    the project folders that identify this repository.
    """
    for candidate in (start_path, *start_path.parents):
        has_data_folder = (candidate / "data").exists()
        has_notebooks_folder = (candidate / "notebooks").exists()

        if has_data_folder and has_notebooks_folder:
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure the notebook is opened inside market-intelligence-pipeline."
    )


project_root = find_project_root(Path.cwd().resolve())

raw_olist_dir = project_root / "data" / "raw" / "olist"

print(f"Project root: {project_root}")
print(f"Raw Olist data: {raw_olist_dir}")

def get_raw_file(file_name: str) -> Path:
    """
    Find a file anywhere inside data/raw/olist/.
    """
    matches = list(raw_olist_dir.rglob(file_name))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {file_name} inside {raw_olist_dir}"
        )

    return matches[0]

orders = pd.read_csv(
    get_raw_file("olist_orders_dataset.csv"),
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_items = pd.read_csv(
    get_raw_file("olist_order_items_dataset.csv"),
    parse_dates=[
        "shipping_limit_date",
    ],
)


print("orders shape:", orders.shape)
print("order_items shape:", order_items.shape)

orders.head()

Project root: C:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline
Raw Olist data: C:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline\data\raw\olist
orders shape: (99441, 8)
order_items shape: (112650, 7)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [3]:
order_items['gross_item_value'] = order_items['price'] + order_items['freight_value']
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,gross_item_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04


In [6]:
order_value = order_items.groupby('order_id', as_index=False).agg(
    gross_order_value=('gross_item_value','sum'),
    item_count=('order_item_id','count'),
    average_item_value=('gross_item_value','mean')
)
order_value.head()

,order_id,gross_order_value,item_count,average_item_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,218.04


In [ ]:
# let's validate the uniqueness, order id should be unique for orders and order_value
print("orders key unique:")
print(orders["order_id"].is_unique)

print("order_items key unique:")
print(order_items["order_id"].is_unique)

print("order_value key unique:")
print(order_value["order_id"].is_unique)

orders key unique:
True
order_items key unique:
False
order_value key unique:
True


In [8]:
assert orders["order_id"].is_unique
assert not order_items["order_id"].is_unique
assert order_value["order_id"].is_unique

In [9]:
# lets check if order items have matching order ids
orders_items_raw = orders.merge(
    order_items,
    on="order_id",
    how="left",
    validate="one_to_many",
    indicator="raw_item_match",
)

In [11]:
print("Orders rows:", len(orders))
print("Raw joined rows:", len(orders_items_raw))
print("Unique orders after raw join:", orders_items_raw["order_id"].nunique())

Orders rows: 99441
Raw joined rows: 113425
Unique orders after raw join: 99441


In [16]:
orders_without_items = orders_items_raw.loc[
    orders_items_raw["raw_item_match"].eq("left_only")
]
orders_without_items.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,gross_item_value,raw_item_match
306,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaT,NaT,2017-12-05,NaN,NaN,NaN,NaT,NaN,NaN,NaN,left_only
671,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,2018-01-31 11:31:37,2018-01-31 14:23:50,NaT,NaT,2018-02-16,NaN,NaN,NaN,NaT,NaN,NaN,NaN,left_only
791,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaT,NaT,2017-09-05,NaN,NaN,NaN,NaT,NaN,NaN,NaN,left_only
850,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaT,NaT,2018-02-06,NaN,NaN,NaN,NaT,NaN,NaN,NaN,left_only
1294,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12,NaN,NaN,NaN,NaT,NaN,NaN,NaN,left_only


In [17]:
orders_enriched = orders.merge(
    order_value,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="item_match",
)

In [ ]:
# orders_without_items = orders_enriched.loc[
#     orders_enriched["item_match"].eq("left_only"),
#     [
#         "order_id",
#         "order_status",
#         "order_purchase_timestamp",
#     ],
# ]

In [18]:
assert len(orders_enriched) == len(orders)

assert (
    orders_enriched["order_id"].nunique()
    == orders["order_id"].nunique()
)

assert orders_enriched["order_id"].is_unique

In [19]:
orders_enriched["gross_order_value_filled"] = (
    orders_enriched["gross_order_value"].fillna(0)
)

In [20]:
orders_without_items = orders_enriched.loc[
    orders_enriched["item_match"].eq("left_only"),
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
    ],
]

missing_items_by_status = (
    orders_without_items
    .groupby("order_status", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
    )
    .sort_values(
        "order_count",
        ascending=False,
    )
)
missing_items_by_status.head()

,order_status,order_count
4,unavailable,603
0,canceled,164
1,created,5
2,invoiced,2
3,shipped,1


In [21]:
orders_enriched["purchase_month"] = (
    orders_enriched["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype("string")
)

delivered_orders = orders_enriched.loc[
    orders_enriched["order_status"].eq("delivered")
].copy()

monthly_delivered_metrics = (
    delivered_orders
    .groupby("purchase_month", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_gross_order_value=("gross_order_value", "sum"),
        average_gross_order_value=("gross_order_value", "mean"),
        average_item_count=("item_count", "mean"),
    )
    .sort_values("purchase_month")
)
monthly_delivered_metrics.head()

,purchase_month,order_count,total_gross_order_value,average_gross_order_value,average_item_count
0,2016-09,1,143.46,143.460000,3.000000
1,2016-10,265,46490.66,175.436453,1.181132
2,2016-12,1,19.62,19.620000,1.000000
3,2017-01,750,127482.37,169.976493,1.217333
4,2017-02,1653,271239.32,164.089123,1.124017
